# DeepSeek-VL2-Tiny on FineSightBench (HF Hub)

Evaluate **deepseek-ai/deepseek-vl2-tiny** on the `Volavion/FineSightBench` dataset (downloaded from the Hugging Face Hub).

**Pipeline**
1. Load dataset from HF Hub (cached under `~/.cache/huggingface/datasets/`).
2. Download / cache the model from HF Hub (no local paths).
3. Run accuracy benchmark on a subset of samples.
4. Visualize per-token last-layer attention on one sample, saving a GIF.


In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import torch
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, HTML, display


# ── 1. Download / load the dataset from Hugging Face Hub ─────────────────────
from datasets import load_dataset

# Force local repo imports first to avoid stale site-packages version.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "finesightbench").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT))
for _m in [m for m in list(sys.modules) if m == "finesightbench" or m.startswith("finesightbench.")]:
    sys.modules.pop(_m, None)


def _parse_simple_dotenv(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        values[key] = value
    return values


def _load_hf_token() -> tuple[str, str]:
    env_token = (
        os.getenv("HF_TOKEN")
        or os.getenv("HUGGINGFACE_HUB_TOKEN")
        or os.getenv("HUGGINGFACE_TOKEN")
        or os.getenv("ACCESS_TOKEN")
    )
    if env_token:
        os.environ.setdefault("HF_TOKEN", env_token)
        os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", env_token)
        return env_token, "environment"

    for name in (".env", "env"):
        env_path = REPO_ROOT / name
        if env_path.exists():
            values = _parse_simple_dotenv(env_path)
            token = (
                values.get("HF_TOKEN")
                or values.get("HUGGINGFACE_HUB_TOKEN")
                or values.get("HUGGINGFACE_TOKEN")
                or values.get("ACCESS_TOKEN")
            )
            if token:
                os.environ["HF_TOKEN"] = token
                os.environ["HUGGINGFACE_HUB_TOKEN"] = token
                return token, str(env_path)

    return "", "not found (.env/env); public-only access"


HF_TOKEN, HF_TOKEN_SOURCE = _load_hf_token()

from finesightbench.evaluation.framework import overlay_attention

MODEL_NAME    = "deepseek-vl2-small"
HF_DATASET_ID = "Volavion/FineSightBench"
OUTPUT_DIR    = REPO_ROOT / "outputs" / "vlm_eval_hf" / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"Output: {OUTPUT_DIR}")
print(f"HF token source: {HF_TOKEN_SOURCE}")

Device: cuda
GPU   : NVIDIA RTX A6000
Output: /home/snt/projects_lujun/FineSightBench/outputs/vlm_eval_hf/deepseek-vl2-small
HF token source: not found (.env/env); public-only access


In [2]:
print(f"Loading dataset: {HF_DATASET_ID}")
ds = load_dataset(HF_DATASET_ID, token=HF_TOKEN or None)   # caches under ~/.cache/huggingface/datasets
print(ds)

print("\nPerception sample 0:")
s0 = ds["perception"][0]
print(f"  image_id  : {s0['image_id']}")
print(f"  task_type : {s0['task_type']}")
print(f"  difficulty: {s0['difficulty']}")
print(f"  question  : {s0['question'][:100]}...")
print(f"  answer    : {s0['answer']}")

Loading dataset: Volavion/FineSightBench
DatasetDict({
    perception: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 4200
    })
    reasoning: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 3920
    })
})

Perception sample 0:
  image_id  : perception_letter_4px_00000
  task_type : letter_recognition
  difficulty: extreme
  question  : What letter is displayed in the image? Answer in JSON format only: {"letter": "<X>"} where <X> is on...
  answer    : {"letter": "A"}


In [3]:
# ── 2. Download / load model + processor from HF Hub ────────────────────────
# DeepSeek-VL2's pip package targets transformers 4.x. We monkey-patch missing
# symbols, wrap `LlamaAttention.forward` to accept the legacy signature, and
# manually load the checkpoint (transformers 5's custom-code loader silently
# skips weights for this model).
import torch

import transformers.utils.import_utils as _iu
import transformers.pytorch_utils     as _pu
import transformers.models.llama.modeling_llama as _llama
if not hasattr(_iu, "is_torch_fx_available"):
    _iu.is_torch_fx_available = lambda: False
if not hasattr(_pu, "is_torch_greater_or_equal_than_1_13"):
    _pu.is_torch_greater_or_equal_than_1_13 = True
if not hasattr(_llama, "LlamaFlashAttention2"):
    _llama.LlamaFlashAttention2 = _llama.LlamaAttention

from transformers.cache_utils import DynamicCache as _DynCache
if not hasattr(_DynCache, "seen_tokens"):
    _DynCache.seen_tokens = property(
        lambda self: self.get_seq_length() if hasattr(self, "get_seq_length") else 0
    )
if not hasattr(_DynCache, "get_max_length"):
    _DynCache.get_max_length = lambda self: None
if not hasattr(_DynCache, "get_usable_length"):
    _DynCache.get_usable_length = lambda self, new_seq_length, layer_idx=0: (
        self.get_seq_length(layer_idx) if hasattr(self, "get_seq_length") else 0
    )
if not hasattr(_DynCache, "from_legacy_cache"):
    _DynCache.from_legacy_cache = classmethod(lambda cls, past: cls())

# ── Wrap transformers-5 LlamaAttention.forward so deepseek_vl2 can call it
# with the transformers-4 signature (position_ids + 3-tuple return).
from transformers.models.llama.modeling_llama import (
    LlamaAttention as _LlamaAttn, LlamaRotaryEmbedding as _LlamaRoPE,
)
_orig_llama_fwd = _LlamaAttn.forward

def _patched_llama_forward(
    self, hidden_states, position_embeddings=None, attention_mask=None,
    past_key_values=None, position_ids=None, past_key_value=None,
    output_attentions=False, use_cache=False, cache_position=None, **kwargs,
):
    if position_embeddings is None:
        if not hasattr(self, "_fs_rope"):
            self._fs_rope = _LlamaRoPE(self.config).to(hidden_states.device)
        if position_ids is None:
            B, L = hidden_states.shape[:2]
            position_ids = torch.arange(L, device=hidden_states.device).unsqueeze(0).expand(B, -1)
        cos, sin = self._fs_rope(hidden_states, position_ids)
        position_embeddings = (cos, sin)
    # drop the unsupported kwargs
    result = _orig_llama_fwd(
        self, hidden_states,
        position_embeddings=position_embeddings,
        attention_mask=attention_mask,
        past_key_values=past_key_values if past_key_values is not None else past_key_value,
    )
    out, attn_w = result[0], result[1] if len(result) > 1 else None
    return out, attn_w, past_key_value

_LlamaAttn.forward = _patched_llama_forward

from transformers import GenerationMixin
from deepseek_vl2.models import DeepseekVLV2ForCausalLM, DeepseekVLV2Processor
from deepseek_vl2.models.modeling_deepseek import DeepseekV2ForCausalLM

DeepseekVLV2ForCausalLM.all_tied_weights_keys = {}
DeepseekV2ForCausalLM.all_tied_weights_keys   = {}
for _cls in (DeepseekV2ForCausalLM, DeepseekVLV2ForCausalLM):
    if GenerationMixin not in _cls.__mro__:
        _cls.__bases__ = (GenerationMixin,) + _cls.__bases__

MODEL_ID = "deepseek-ai/deepseek-vl2-tiny"
print(f"Loading processor: {MODEL_ID}")
vl_processor = DeepseekVLV2Processor.from_pretrained(MODEL_ID, token=HF_TOKEN or None)
tokenizer    = vl_processor.tokenizer

print(f"Loading model (eager attention, bfloat16) ...")
model = DeepseekVLV2ForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN or None,
    dtype=torch.bfloat16,
    device_map=DEVICE,
    trust_remote_code=True,
    attn_implementation="eager",
)

# transformers 5 silently skips weight loading for this custom model → reload
from safetensors.torch import load_file
from huggingface_hub import snapshot_download
_snap = snapshot_download(MODEL_ID, token=HF_TOKEN or None, allow_patterns=["*.safetensors", "*.json"])
import glob as _glob, os as _os
_shards = sorted(_glob.glob(_os.path.join(_snap, "model*.safetensors")))
print(f"Reloading weights from {len(_shards)} shard(s) ...")
_state = {}
for _s in _shards:
    _state.update(load_file(_s))
_missing, _unexpected = model.load_state_dict(_state, strict=False)
print(f"  missing={len(_missing)} unexpected={len(_unexpected)}")
model.to(dtype=torch.bfloat16)
model.eval()
print(f"Model class : {type(model).__name__}")
print(f"Params      : {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")

MAX_PER_TASK = 2
SEED         = 42
import random
from collections import defaultdict
rng = random.Random(SEED)

# Group samples from every split by task_type, then pick up to MAX_PER_TASK
# per task. Each picked item carries its source split so we can still group
# by split later.
_by_task: dict[str, list[tuple[str, dict]]] = defaultdict(list)
for split_name in ("perception", "reasoning"):
    for item in ds[split_name]:
        _by_task[item["task_type"]].append((split_name, item))

subset: dict[str, list[dict]] = {"perception": [], "reasoning": []}
for task, items in _by_task.items():
    rng.shuffle(items)
    chosen = items[:MAX_PER_TASK]
    for split_name, item in chosen:
        subset[split_name].append(item)

print("Samples per task_type (2 each):")
for task in sorted(_by_task):
    n = sum(1 for s in subset["perception"] + subset["reasoning"]
            if s["task_type"] == task)
    print(f"  {task:28s} {n}")
print(f"\nTotal selected: perception={len(subset['perception'])}, "
      f"reasoning={len(subset['reasoning'])}")

Python version is above 3.10, patching the collections module.
Loading processor: deepseek-ai/deepseek-vl2-tiny


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Add pad token = ['<｜▁pad▁｜>'] to the tokenizer
<｜▁pad▁｜>:2
Add image token = ['<image>'] to the tokenizer
<image>:128815
Add grounding-related tokens = ['<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>'] to the tokenizer with input_ids
<|ref|>:128816
<|/ref|>:128817
<|det|>:128818
<|/det|>:128819
<|grounding|>:128820
Add chat tokens = ['<|User|>', '<|Assistant|>'] to the tokenizer with input_ids
<|User|>:128821
<|Assistant|>:128822

Loading model (eager attention, bfloat16) ...


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 41645.57it/s]


Reloading weights from 1 shard(s) ...
  missing=0 unexpected=0
Model class : DeepseekVLV2ForCausalLM
Params      : 3.37 B
Samples per task_type (2 each):
  animal_recognition           2
  block_recognition            2
  blur_chain                   2
  chain_reasoning              2
  color_block_recognition      2
  comparison_chain             2
  counting_chain               2
  letter_recognition           2
  shape_recognition            2
  text_counting_chain          2
  text_reading_chain           2
  text_recognition             2

Total selected: perception=12, reasoning=12


In [4]:
# ── 3. DeepSeek-VL2 inference helpers ────────────────────────────────────────
# `transformers 5` broke `.generate(inputs_embeds=...)` for the pip version of
# deepseek_vl2 (empty `input_ids` reach the decoder). We do manual greedy
# decoding with forward passes over inputs_embeds directly.
def _build_deepseek_inputs(image: Image.Image, question: str):
    conversation = [
        {"role": "<|User|>", "content": f"<image>\n{question}", "images": [image]},
        {"role": "<|Assistant|>", "content": ""},
    ]
    prepare = vl_processor(
        conversations=conversation,
        images=[image],
        force_batchify=True,
        system_prompt="",
    )
    return prepare.to(DEVICE, dtype=torch.bfloat16)


@torch.no_grad()
def deepseek_predict(image: Image.Image, question: str, max_new_tokens: int = 512) -> str:
    prepare       = _build_deepseek_inputs(image, question)
    inputs_embeds = model.prepare_inputs_embeds(**prepare)
    attn_mask     = prepare.attention_mask
    embed_layer   = model.language.get_input_embeddings()
    eos_id        = tokenizer.eos_token_id

    generated: list[int] = []
    cur_embeds, cur_mask = inputs_embeds, attn_mask
    for _ in range(max_new_tokens):
        out = model.language.model(
            inputs_embeds=cur_embeds, attention_mask=cur_mask,
            use_cache=False, return_dict=True,
        )
        logits  = model.language.lm_head(out.last_hidden_state[:, -1, :])
        next_id = int(logits.argmax(dim=-1).item())
        if next_id == eos_id:
            break
        generated.append(next_id)
        next_emb = embed_layer(torch.tensor([[next_id]], device=DEVICE)).to(cur_embeds.dtype)
        cur_embeds = torch.cat([cur_embeds, next_emb], dim=1)
        cur_mask   = torch.cat(
            [cur_mask, torch.ones((1, 1), dtype=cur_mask.dtype, device=DEVICE)], dim=1,
        )
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# ── 4. JSON evaluation: schemas, gold-answer parser ──────────────────────────
# The HF dataset already contains structured gold answers.
# We evaluate against the expected JSON schema per task_type.
import re as _re
from finesightbench.evaluation import (
    FieldSpec, ORDERED_LIST, MAPPING,
    evaluate_json_prediction, aggregate_json_results,
)

# --- JSON schemas per task_type ---------------------------------------------
TASK_SCHEMAS = {
    "letter_recognition":      [FieldSpec("letter")],
    "animal_recognition":      [FieldSpec("animal")],
    "block_recognition":       [FieldSpec("present")],
    "color_block_recognition": [FieldSpec("color")],
    "shape_recognition":       [FieldSpec("shape")],
    "chain_reasoning":         [FieldSpec("objects", kind=ORDERED_LIST)],
    "comparison_chain":        [FieldSpec("objects", kind=ORDERED_LIST)],
    "counting_chain":          [FieldSpec("counts", kind=MAPPING), FieldSpec("total")],
    "blur_chain":              [FieldSpec("counts", kind=MAPPING), FieldSpec("total")],
}

# --- Build ground-truth object from HF dataset gold answer -------------------
# The dataset's `answer` field is already a scalar or JSON string.
# Try to parse as JSON; if that fails, treat as scalar text.
def build_gt_obj(task_type: str, gold_text: str) -> dict:
    """
    Parse gold answer into expected gt_obj structure.
    If gold_text is already valid JSON, parse and use it directly.
    Otherwise, construct based on task_type.
    """
    gold_text = str(gold_text).strip()
    
    # Try to parse as JSON first
    try:
        parsed = json.loads(gold_text)
        # If it's already a dict with the expected structure, return as-is
        if isinstance(parsed, dict):
            return parsed
    except (json.JSONDecodeError, ValueError):
        pass
    
    # Fallback: construct based on task_type
    if task_type == "letter_recognition":
        return {"letter": gold_text}
    elif task_type == "animal_recognition":
        return {"animal": gold_text.lower()}
    elif task_type == "block_recognition":
        return {"present": gold_text.lower()}
    elif task_type == "color_block_recognition":
        return {"color": gold_text.lower()}
    elif task_type == "shape_recognition":
        return {"shape": gold_text.lower()}
    elif task_type in ("chain_reasoning", "comparison_chain"):
        # Try to parse as JSON array; if not, split by comma
        try:
            parsed = json.loads(gold_text)
            if isinstance(parsed, list):
                return {"objects": [str(x).lower() for x in parsed]}
            elif isinstance(parsed, dict) and "objects" in parsed:
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass
        # Fallback: split by comma
        items = [p.strip().lower() for p in gold_text.split(",") if p.strip()]
        return {"objects": items}
    elif task_type in ("counting_chain", "blur_chain"):
        # Try to parse as JSON; fallback to manual parsing
        try:
            parsed = json.loads(gold_text)
            if isinstance(parsed, dict) and "counts" in parsed:
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass
        # Fallback: manual parse "N color, M color; total: X"
        counts_part, _, total_part = gold_text.partition(";")
        counts: dict[str, int] = {}
        for chunk in counts_part.split(","):
            chunk = chunk.strip()
            if not chunk:
                continue
            m = _re.match(r"(\d+)\s+(.+)", chunk)
            if m:
                counts[m.group(2).strip().lower()] = int(m.group(1))
        tm = _re.search(r"total\s*:\s*(\d+)", total_part)
        total = int(tm.group(1)) if tm else sum(counts.values())
        return {"counts": counts, "total": total}
    
    # Fallback for unknown tasks
    return {"answer": gold_text}

# Quick sanity check
_seen = {}
for split in ("perception", "reasoning"):
    for s in subset[split]:
        _seen.setdefault(s["task_type"], s)
for t, s in _seen.items():
    gold_obj = build_gt_obj(t, s["answer"])
    print(f"{t:28s} -> {gold_obj}")


letter_recognition           -> {'letter': 'D'}
animal_recognition           -> {'animal': 'fish'}
block_recognition            -> {'present': 'yes'}
color_block_recognition      -> {'color': 'gray'}
shape_recognition            -> {'shape': 'triangle'}
text_recognition             -> {'word': 'LOUD'}
chain_reasoning              -> {'objects': ['orange dot', 'orange dot']}
comparison_chain             -> {'objects': ['brown dot', 'green dot', 'purple dot', 'black dot', 'pink dot', 'gray dot', 'cyan dot', 'red dot']}
counting_chain               -> {'counts': {'green': 1, 'red': 1}, 'total': 2}
blur_chain                   -> {'counts': {'bird': 1, 'fish': 1}, 'total': 2}
text_reading_chain           -> {'words': ['LION', 'BEAR', 'BOLD', 'BANK', 'CITY', 'PARK', 'EXIT', 'PLAY']}
text_counting_chain          -> {'total': 6, 'with_letter': 2}


In [5]:
# ── 5. Run the JSON benchmark on the subset ──────────────────────────────────
import time

rows = []
t0 = time.time()
for split in ("perception", "reasoning"):
    for i, s in enumerate(subset[split]):
        img = s["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))
        img = img.convert("RGB")

        task = s["task_type"]
        # Use the question as-is; dataset already includes expected format in the question
        prompt = s["question"]

        try:
            raw_pred = deepseek_predict(img, prompt)
            infer_error = ""
        except Exception as e:
            raw_pred = ""
            infer_error = f"{type(e).__name__}: {e}"

        schema = TASK_SCHEMAS.get(task, [FieldSpec("answer")])
        gt_obj = build_gt_obj(task, s["answer"])
        result = evaluate_json_prediction(raw_pred, gt_obj, schema)

        rows.append({
            "split": split,
            "task_type": task,
            "image_id": s["image_id"],
            "gold_text": s["answer"],
            "gold_obj": gt_obj,
            "raw_pred": raw_pred,
            "infer_error": infer_error,
            "json_eval": result.to_dict(),
        })

        print(
            f"  [{split}] {i+1}/{len(subset[split])} task={task} "
            f"parse_ok={result.parse_ok} score={result.overall_score:.2f} "
            f"strict={result.all_fields_matched}"
        )

elapsed = time.time() - t0
print(f"\nElapsed: {elapsed:.1f}s")


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  [perception] 1/12 task=letter_recognition parse_ok=True score=1.00 strict=True
  [perception] 2/12 task=letter_recognition parse_ok=True score=1.00 strict=True
  [perception] 3/12 task=animal_recognition parse_ok=True score=1.00 strict=True
  [perception] 4/12 task=animal_recognition parse_ok=True score=1.00 strict=True
  [perception] 5/12 task=block_recognition parse_ok=True score=1.00 strict=True
  [perception] 6/12 task=block_recognition parse_ok=True score=0.00 strict=False
  [perception] 7/12 task=color_block_recognition parse_ok=True score=1.00 strict=True
  [perception] 8/12 task=color_block_recognition parse_ok=True score=1.00 strict=True
  [perception] 9/12 task=shape_recognition parse_ok=True score=1.00 strict=True
  [perception] 10/12 task=shape_recognition parse_ok=True score=1.00 strict=True
  [perception] 11/12 task=text_recognition parse_ok=True score=0.00 strict=False
  [perception] 12/12 task=text_recognition parse_ok=True score=0.00 strict=False
  [reasoning] 1/12 t

In [6]:
# ── 6. Aggregate: accuracy, hallucination rate, per-field/positional stats ──
summary = aggregate_json_results(rows, group_keys=("split", "task_type"))

print("=== Overall ===")
print(f"  n                    : {summary['n']}")
print(f"  strict accuracy      : {summary['strict_accuracy']:.3%}")
print(f"  mean overall score   : {summary['mean_overall_score']:.3f}")
print(f"  hallucinations       : {summary['hallucinations']} "
      f"(rate {summary['hallucination_rate']:.3%})")

print("\n=== Per field ===")
for fname, fs in summary["per_field"].items():
    extra = ""
    if "positional_accuracy" in fs:
        extra = (f"  positional_acc={fs['positional_accuracy']:.3%} "
                 f"({fs['positional_correct']}/{fs['positional_expected_total']})")
    print(f"  {fname:10s} n={fs['n']:>3d} "
          f"exact={fs['accuracy']:.3%} mean_score={fs['mean_score']:.3f}{extra}")

print("\n=== Per split × task ===")
for g, gs in summary["groups"].items():
    print(f"  {g:50s} n={gs['n']:>3d} strict={gs['strict_accuracy']:.3%} "
          f"mean={gs['mean_overall_score']:.3f} halluc={gs['hallucination_rate']:.3%}")

(OUTPUT_DIR / "predictions.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False, default=str) for r in rows)
)
(OUTPUT_DIR / "report.json").write_text(json.dumps({
    "model":   MODEL_NAME,
    "dataset": HF_DATASET_ID,
    "subset":  {"max_per_task": MAX_PER_TASK,
                "perception": len(subset["perception"]),
                "reasoning":  len(subset["reasoning"])},
    "seed":    SEED,
    "elapsed_seconds": elapsed,
    "summary": summary,
}, indent=2, ensure_ascii=False, default=str))
print(f"\nPredictions -> {OUTPUT_DIR/'predictions.jsonl'}")
print(f"Report      -> {OUTPUT_DIR/'report.json'}")


=== Overall ===
  n                    : 18
  strict accuracy      : 61.111%
  mean overall score   : 0.669
  hallucinations       : 0 (rate 0.000%)

=== Per field ===
  animal     n=  2 exact=100.000% mean_score=1.000
  color      n=  2 exact=100.000% mean_score=1.000
  counts     n=  4 exact=50.000% mean_score=0.917
  letter     n=  2 exact=100.000% mean_score=1.000
  objects    n=  4 exact=0.000% mean_score=0.050  positional_acc=4.348% (2/22)
  present    n=  2 exact=50.000% mean_score=0.500
  shape      n=  2 exact=100.000% mean_score=1.000
  total      n=  4 exact=50.000% mean_score=0.500

=== Per split × task ===
  split=perception | task_type=animal_recognition    n=  2 strict=100.000% mean=1.000 halluc=0.000%
  split=perception | task_type=block_recognition     n=  2 strict=50.000% mean=0.500 halluc=0.000%
  split=perception | task_type=color_block_recognition n=  2 strict=100.000% mean=1.000 halluc=0.000%
  split=perception | task_type=letter_recognition    n=  2 strict=100.00

In [7]:
rows

[{'split': 'perception',
  'task_type': 'letter_recognition',
  'image_id': 'perception_letter_24px_00415',
  'gold_text': '{"letter": "D"}',
  'gold_obj': {'letter': 'D'},
  'raw_pred': '{"letter": "D"}',
  'infer_error': '',
  'json_eval': {'parse_ok': True,
   'parse_error': '',
   'raw_pred': '{"letter": "D"}',
   'pred_obj': {'letter': 'D'},
   'field_results': {'letter': {'kind': 'scalar',
     'expected': 'D',
     'predicted': 'D',
     'missing': False,
     'matched': True,
     'score': 1.0}},
   'overall_score': 1.0,
   'all_fields_matched': True,
   'hallucination': False}},
 {'split': 'perception',
  'task_type': 'letter_recognition',
  'image_id': 'perception_letter_32px_00508',
  'gold_text': '{"letter": "L"}',
  'gold_obj': {'letter': 'L'},
  'raw_pred': '{"letter": "L"}',
  'infer_error': '',
  'json_eval': {'parse_ok': True,
   'parse_error': '',
   'raw_pred': '{"letter": "L"}',
   'pred_obj': {'letter': 'L'},
   'field_results': {'letter': {'kind': 'scalar',
     '